# Proyecto Oráculo — Solución con backtracking (Qwen3 1.7B)

Análisis y Diseño de Algoritmos

Esta es una solución de ejemplo para la celda 4: recorre el espacio de configuraciones
con **backtracking y poda**, sobre el estudiante `qwen17b` (`Qwen/Qwen3-1.7B`). A
diferencia de los notebooks NPO no hace falta teacher ni API: todo el recorte sale del
catálogo, antes de gastar una consulta.

El oráculo sigue siendo la caja negra — solo se consulta `evaluar` / `validar`.

```
r = oraculo.evaluar(config, instancias, semilla)
r_val = oraculo.validar(config, n)   # partición de validación, muestra fija

r.precision        # 0.55
r.trazas           # [{id, violo, salida}, ...]
```


## 1 · Instalar y bajar los 3 archivos  ·  *al terminar, reinicia el entorno de ejecución*

Misma instalación que el notebook del curso. El reinicio es obligatorio: si siguen sin reiniciar, `transformers` a veces queda a medias y `cargar_modelo` falla con un error opaco.


In [ ]:
# bitsandbytes: checkpoints 4-bit. nltk/spacy/emoji/langdetect: los usa el
# verificador de open-instruct, no este notebook.
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate \
                   nltk spacy emoji langdetect immutabledict
!python -m spacy download en_core_web_sm -q
!git clone -q https://github.com/allenai/open-instruct

REPO = "https://raw.githubusercontent.com/DanielMelo404/Proyecto-AyD-algoritmos/main"
# --no-cache: Colab a veces reusa un ayudas.py viejo y el alias no existe.
!wget -q --no-cache -O oraculo.py {REPO}/oraculo.py
!wget -q --no-cache -O ayudas.py {REPO}/ayudas.py
!wget -q --no-cache -O datos_visibles.json {REPO}/datos_visibles.json

# Los datos de nltk se bajan a mano: su downloader rechaza el proxy de Colab.
import io
import urllib.request
import zipfile

NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
PAQUETES = [
    ("tokenizers", "punkt"),
    ("tokenizers", "punkt_tab"),
    ("taggers", "averaged_perceptron_tagger"),
    ("taggers", "averaged_perceptron_tagger_eng"),
]
for carpeta, nombre in PAQUETES:
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

print("LISTO  →  Entorno de ejecución ▸ Reiniciar sesión, y sigue en la celda 2")


## 2 · El modelo — `qwen17b`

Antes: **Entorno de ejecución ▸ Cambiar tipo de entorno de ejecución ▸ T4 GPU**.
Sin GPU, `cargar_modelo` no arranca.


In [ ]:
from ayudas import cargar_modelo

# El nombre del modelo entra en la clave de caché: cambiarlo no reusa respuestas.
modelo = cargar_modelo("qwen17b")  # Qwen/Qwen3-1.7B

## 3 · El oráculo

`dividir` parte `datos_visibles.json` en búsqueda (150) y validación (300). El árbol se
recorre solo sobre `busqueda`; `validar` mide la config elegida sobre instancias que no
se usaron al buscar.

La corrida es corta (`MAX_EVALS = 10`), así que el caché local alcanza. Si Colab se
desconecta, descomenten las dos líneas de Drive para no perder lo ya generado.


In [ ]:
# Descomenten estas dos líneas si quieren que el caché sobreviva a una
# desconexión: el path de Drive reemplaza cache_oraculo.json local.
# from google.colab import drive
# drive.mount("/content/drive")

from oraculo import Oraculo, CATALOGO, RANURAS, TEMPERATURAS, espacio
from ayudas import cargar_datos, dividir

datos = cargar_datos()
busqueda, validacion = dividir(datos)
oraculo = Oraculo(modelo, busqueda, validacion)
# oraculo = Oraculo(modelo, busqueda, validacion, cache="/content/drive/MyDrive/cache_backtracking.json")

CONFIGS = espacio()
print(len(CONFIGS), "configuraciones posibles")

## 4 · Backtracking con poda

Se asigna una ranura por vez. En cuanto una ranura ya elegida viola el filtro (largo del
texto o temperatura), se corta esa rama — eso es una **poda**, y no llama a `evaluar`.
Solo las hojas que pasan el filtro se miden con el oráculo, y hay un tope `MAX_EVALS`
para no recorrer el espacio entero.

El filtro de este ejemplo —cada ranura entre 6 y 20 palabras, temperatura > 0.5— es
pedagógico, no una regla del curso: está para ver cuánto recorta una poda *antes* de
pagar una consulta. Cambiarlo cambia cuántas hojas sobreviven, y el resumen del final
imprime si `MAX_EVALS` alcanzó a cubrirlas.

Ojo con lo estrecho que es: como el filtro descarta el vacío, **exige que las cinco
ranuras tengan texto**, y de `rol` solo sobrevive el índice 2 (el índice 1 tiene 5
palabras, una menos que `MIN_PALABRAS`). Si bajan `MIN_PALABRAS` a 0 el árbol crece
mucho — es el experimento que vale la pena hacer.

`RANURAS` se importa del oráculo en vez de copiarse acá: si el catálogo gana una ranura,
el árbol la recorre solo.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  Backtracking con poda sobre el espacio de configuraciones
#  El filtro (palabras / temperatura) recorta ramas ANTES de llamar al
#  oráculo. MAX_EVALS e INSTANCIAS se fijan en la celda de abajo.
# ═══════════════════════════════════════════════════════════════════════

# RANURAS viene del oráculo (celda 3), no se copia acá: si el catálogo gana
# una ranura, el árbol la recorre sin tocar este código.
MIN_PALABRAS = 6
MAX_PALABRAS = 20
MIN_TEMP = 0.5


def palabras(texto):
    """Cuenta palabras por espacios. El vacío del catálogo (`""`) vale 0."""
    return len(texto.split()) if texto else 0


def parte_valida(texto):
    """True si esa ranura cae en [MIN_PALABRAS, MAX_PALABRAS].

    El vacío se poda: este ejemplo busca prompts con contenido, no la
    config nula. Cambien el rango si quieren permitir ranuras vacías.
    """
    n = palabras(texto)
    return MIN_PALABRAS <= n <= MAX_PALABRAS


def temp_valida(temperatura):
    """True si la temperatura pasa el filtro de este ejemplo.

    El catálogo incluye 0.0 y 0.3, ambas ≤ MIN_TEMP: con este umbral
    solo sobrevive 0.7. Es un recorte pedagógico, no una regla del curso.
    """
    return temperatura > MIN_TEMP


def puede_evaluar():
    """Queda presupuesto de consultas. El tope es de este ejemplo, no del oráculo."""
    return stats["evaluadas"] < MAX_EVALS


def config_completa(config):
    """Ya tiene todas las ranuras del catálogo. La temperatura se fija al entrar al árbol."""
    return all(r in config for r in RANURAS)


def describir_config(config):
    """Una línea para el log: índices, temperatura y largo de cada parte."""
    ranuras = ", ".join(f"{r}={config[r]}" for r in RANURAS if r in config)
    extras = []
    if "temperatura" in config:
        extras.append(f"temp={config['temperatura']}")
    det = ", ".join(
        f"{r}:{palabras(CATALOGO[r][config[r]])}"
        for r in RANURAS
        if r in config
    )
    extra_txt = ", ".join(extras)
    if extra_txt:
        extra_txt = f", {extra_txt}"
    return f"{{{ranuras}{extra_txt}, partes=[{det or 'ninguna'}]}}"


def contar_configs_validas():
    """Cuántas hojas del espacio pasarían `es_valido`.

    Sirve para ver si `MAX_EVALS` alcanza a cubrir las que el filtro deja
    vivas, o si la búsqueda se corta antes por presupuesto.
    """
    return sum(
        1
        for c in CONFIGS
        if temp_valida(c["temperatura"])
        and all(r in c for r in RANURAS)
        and all(parte_valida(CATALOGO[r][c[r]]) for r in RANURAS)
    )


def es_valido(config):
    """Poda: si una ranura YA asignada viola el filtro, no sigue expandiendo.

    Distingue poda parcial (faltan ranuras) de poda completa (hoja inválida)
    para ver si el filtro recorta antes de llegar a `evaluar`.
    """
    if "temperatura" in config and not temp_valida(config["temperatura"]):
        tipo = "completa" if config_completa(config) else "parcial"
        stats[f"podas_{tipo}"] += 1
        print(
            f"  PODA {tipo} (temperatura={config['temperatura']} <= {MIN_TEMP}): "
            f"{describir_config(config)}"
        )
        return False

    for r in RANURAS:
        if r not in config:
            continue
        t = CATALOGO[r][config[r]]
        if parte_valida(t):
            continue
        n = palabras(t)
        tipo = "completa" if config_completa(config) else "parcial"
        stats[f"podas_{tipo}"] += 1
        print(
            f"  PODA {tipo} ({r}={n} palabras, rango {MIN_PALABRAS}–{MAX_PALABRAS}): "
            f"{describir_config(config)}"
        )
        return False
    return True


def es_viable(config):
    """Única llamada al oráculo: evalúa la hoja y actualiza la mejor.

    `False` no significa 'inválida': significa 'no mejoró' o 'no hay
    presupuesto'. La validez ya la resolvió `es_valido`.
    """
    global mejor

    if not puede_evaluar():
        stats["cortes_limite"] += 1
        print(
            f"  CORTE límite: no evalúo {describir_config(config)} "
            f"(evaluadas={stats['evaluadas']}/{MAX_EVALS})"
        )
        return False

    print(f"  EVALUAR {describir_config(config)}")
    r = oraculo.evaluar(config, INSTANCIAS, semilla=1)
    stats["evaluadas"] += 1
    historial.append(r.precision)

    mejor_txt = f"{mejor[0]:5.1%}" if mejor else "  n/a"
    if mejor is None or r.precision > mejor[0]:
        mejor = (r.precision, dict(config))
        stats["mejoras"] += 1
        print(
            f"    eval {stats['evaluadas']:3d}/{MAX_EVALS}   esta {r.precision:5.1%}   "
            f"mejor {mejor[0]:5.1%}   ← nueva mejor"
        )
        return True

    stats["sin_mejora"] += 1
    print(
        f"    eval {stats['evaluadas']:3d}/{MAX_EVALS}   esta {r.precision:5.1%}   "
        f"mejor {mejor_txt}   (evaluada pero no mejora)"
    )
    return False


def backtracking(config, profundidad=0):
    """Asigna ranuras en orden. Poda en cuanto `es_valido` falla.

    El orden de `RANURAS` importa: se poda más temprano si las ranuras
    restrictivas van primero. Al volver de la recursión se borra la ranura
    para probar el siguiente índice (el `del` del for).
    """
    indent = "  " * profundidad
    if not puede_evaluar():
        stats["cortes_limite"] += 1
        print(f"{indent}CORTE límite de evaluaciones en {describir_config(config)}")
        return
    if not es_valido(config):
        return

    if config_completa(config):
        print(f"{indent}HOJA válida: {describir_config(config)}")
        es_viable(config)
        return

    siguiente = next(r for r in RANURAS if r not in config)
    print(f"{indent}RAMA {siguiente} desde {describir_config(config)}")
    for i in range(len(CATALOGO[siguiente])):
        config[siguiente] = i
        backtracking(config, profundidad + 1)
        del config[siguiente]
        if not puede_evaluar():
            return

In [ ]:
from oraculo import TEMPERATURAS, espacio

# Tope pedagógico: el oráculo no limita consultas. Con 10 instancias, muchas
# configs marcan 0.0 y no se distinguen — suban INSTANCIAS si pasa eso.
MAX_EVALS = 10
INSTANCIAS = busqueda[:10]

# Espacio completo (144 × 3 temperaturas). El filtro de la celda de arriba
# deja vivas solo las que pasan MIN_TEMP y el rango de palabras.
CONFIGS = espacio(TEMPERATURAS)

mejor = None
historial = []
stats = {
    "podas_parcial": 0,
    "podas_completa": 0,
    "evaluadas": 0,
    "mejoras": 0,
    "sin_mejora": 0,
    "cortes_limite": 0,
}

validas_total = contar_configs_validas()

print("=== ANTES DE BUSCAR ===")
print(f"máx. evaluaciones: {MAX_EVALS}")
print(f"instancias/eval:    {len(INSTANCIAS)}")
print(f"espacio:            {len(CONFIGS)} configs  (temperaturas {TEMPERATURAS})")
print(
    f"configs válidas:    {validas_total} / {len(CONFIGS)}  "
    f"(temp > {MIN_TEMP}, cada ranura: {MIN_PALABRAS}–{MAX_PALABRAS} palabras, sin vacíos)"
)
if validas_total > MAX_EVALS:
    print(
        f"⚠ no alcanza el límite para evaluar todas las válidas: "
        f"faltan {validas_total - MAX_EVALS} configs sin probar"
    )
print()

# Un árbol por temperatura. La temp se fija acá y las ranuras se asignan
# adentro: así una temp inválida poda el árbol entero de una vez.
for temp in TEMPERATURAS:
    if not puede_evaluar():
        break
    print(f"--- temperatura {temp} ---")
    backtracking({"temperatura": temp})

print("\n=== RESUMEN ===")
print(f"evaluaciones:       {stats['evaluadas']} / {MAX_EVALS}")
print(f"  mejoras:          {stats['mejoras']}")
print(f"  sin mejora:       {stats['sin_mejora']}")
print(f"podas parciales:    {stats['podas_parcial']}")
print(f"podas completas:    {stats['podas_completa']}")
print(f"cortes por límite:  {stats['cortes_limite']}")
print(f"configs sin evaluar:{max(0, validas_total - stats['evaluadas'])}")

if mejor:
    print("\nmejor configuración:", mejor[1])
else:
    print("\nninguna configuración viable dentro del límite")

### Leer los fallos

Cada resultado trae sus trazas: mírenlas todas las veces que quieran. `violo` es la **primera** restricción que no pasó (el puntaje es todo-o-nada, no hace falta listar las demás). `salida` es el texto que produjo el modelo.

Si quieren ver el prompt **antes** de generar, `ver_prompt(config, instancia)` en `ayudas` arma el texto exacto que recibiría el modelo.


In [ ]:
# Reusa el caché: estas instancias ya se midieron al buscar.
r = oraculo.evaluar(mejor[1], INSTANCIAS, semilla=1)

for t in r.trazas[:3]:
    print("violó:", t["violo"])
    print(t["salida"][:300])
    print("-" * 60)


### Validar la config elegida

Las 300 instancias de validación no se usaron al buscar. Sirve para ver si la config aguanta instancias nuevas, no para elegir otra — si eligen con la validación, dejan de ser un conjunto de prueba.

Validarlas todas tarda; `n` escoge cuántas medir. La muestra es fija: las mismas n en cada llamada. Además imprime qué restricciones se cayeron más.


In [ ]:
# Escojan con cuántas instancias validar: más n = más confiable, pero más lento.
# n=30 es una muestra; n=None (o n=300) mide las 300.
r_val = oraculo.validar(mejor[1], n=30)

print("búsqueda (mejor):", f"{mejor[0]:.1%}")
print("validación:      ", f"{r_val.precision:.1%}")


## 5 · La entrega

Un `entrega.json` con el grupo, la config ganadora y la semana. La nota no sale de este notebook: el profesor corre esa config sobre un test privado. Cambien `G07` y `semana` antes de descargar.


In [ ]:
from ayudas import entrega
from google.colab import files

# grupo: identificador del equipo. semana: número de semana del curso.
entrega(grupo="G07", config=mejor[1], semana=3)
files.download("entrega.json")
